# PhysDrive ZIP ön kontrolü — CPU, eğitim başlatmaz

23 Kasım 2026. Tam ZIP henüz bu projede incelenmedi. Bu defter arşivin gerçek yapısını ve küçük etiket dosyalarının biçimini kontrol eder; model eğitimi değildir.

**Önce yerel inceleme tercih edilir:** 70 GB ZIP'i açmadan, bilgisayarda aynı betikle envanter oluşturulabilir. Bu defteri kullanmak için 70 GB veriyi yeniden yüklemek zorunda değilsiniz.

Yerel komut (depo kökünde):
```powershell
.venv/Scripts/python.exe scripts/inspect_physdrive_zip.py --zip "C:/Users/Hp/Downloads/PhysDrive(Pub).zip" --out "C:/Users/Hp/Downloads/PhysDrive_onkontrol"
```

Kaggle'da ZIP zaten özel bir input olarak mevcutsa bu defter kullanılabilir. Kaynak veriyi veya çıktıları herkese açık yayımlamayın; Kaggle işleme koşullarını veri sahibinin izniyle netleştirin. Gönderilen OneDrive görüntüleme adresi, doğrulanmış bir anonim indirme adresi değildir. Tarayıcı oturumu Kaggle'a aktarılmaz.
Alternatif: bulut saklama/işleme izni varsa ZIP özel HF dataset deposunda tutulabilir. `HF_REPO` ve `HF_FILE` ayarlanınca 70 GB'ın tamamını Kaggle'a indirmek yerine seek destekli dosya arayüzüyle envanter ve küçük etiketler okunur. Bu yol gerçek PhysDrive dosyasında henüz denenmedi; HF veri aktarımı bloklar hâlinde gerçekleşir.

- Accelerator: **None**. Internet: açık. Yerel ZIP modunda token gerekmez; HF modunda Kaggle Secrets üzerinden `HF_TOKEN` erişimini açın.
- Arşiv diske çıkarılmaz; video gövdeleri okunmaz. En fazla 6 küçük etiket dosyası, her biri en çok 2 MB okunur.
- Kişiler oturum klasörü adlarından tahmin edilmez; BVP örnekleme hızı ve zaman hizalaması varsayılmaz.
- Sonraki adım: envanterden RGB/PPG yapısını, gerçek katılımcı eşlemesini ve tek oturumun zamanlamasını doğrulayıp eğitim yükleyicisini hazırlamak.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
ZIP_PATH = ''  # Yerel/Kaggle input ZIP'i
HF_REPO = ''  # Alternatif: izinli özel HF dataset deposu, ör. kullanici/physdrive-private
HF_FILE = 'PhysDrive(Pub).zip'
assert not (ZIP_PATH and HF_REPO), 'Tek veri kaynağı seçin.'


In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path
if importlib.util.find_spec('scipy') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'], check=True)
if HF_REPO:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=0.27,<2'], check=True)
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
REPO = Path('/tmp/rppg_physdrive_code')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Nisagwn/rppg.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], check=True)


In [ ]:
if not ZIP_PATH and not HF_REPO:
    found = sorted(p for p in Path('/kaggle/input').rglob('*.zip') if 'physdrive' in p.name.lower())
    if len(found) != 1:
        raise RuntimeError(f'Tek PhysDrive ZIP bekleniyor, bulunan: {len(found)}. ZIP_PATH ayarlayın veya yerel incelemeyi kullanın.')
    ZIP_PATH = str(found[0])
if not HF_REPO and not Path(ZIP_PATH).is_file():
    raise FileNotFoundError('ZIP_PATH altında arşiv yok; OneDrive web sayfası bir ZIP dosyası değildir.')
OUT = Path('/kaggle/working/cikti/physdrive_onkontrol')
source = ['--hf-repo', HF_REPO, '--hf-file', HF_FILE] if HF_REPO else ['--zip', ZIP_PATH]
subprocess.run([sys.executable, str(REPO / 'scripts/inspect_physdrive_zip.py'), *source, '--out', str(OUT)], check=True)
print('Ön kontrol tamamlandı. JSON ve CSV çıktılarını inceleyin; eğitim çalıştırılmadı.')
